# Lab 4 - Prompting et RAG
### Introduction aux LLMs - ESGI Paris | Séance 4
---
**Thème du lab :** comparer des stratégies de prompting, puis ancrer un LLM dans des connaissances externes avec un mini pipeline RAG
**Durée estimée :** 45 min - 1h
**Objectifs :**
- Comparer zero-shot, few-shot et Chain-of-Thought sur un petit modèle instruction-tuné
- Construire un pipeline RAG minimal (embeddings + retrieval + génération)
- Observer concrètement quand chaque approche aide (ou pas)

**Prérequis :** Python 3.8+. CPU suffisant (modèles très petits) ; GPU si disponible sur Colab pour aller plus vite.

**Modèle utilisé :** [`HuggingFaceTB/SmolLM2-360M-Instruct`](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct), un petit modèle instruction-tuné (360 millions de paramètres) — assez léger pour tourner rapidement sans crédits GPU.

⚠️ **À garder en tête** : un modèle de 360M de paramètres est *beaucoup* plus petit que les LLMs de production (GPT-4, Claude...). Ses réponses seront parfois imparfaites ou incohérentes — c'est normal, et instructif : ça permet d'observer concrètement les limites de l'échelle sur ces techniques.

## Installation des dépendances

In [1]:
# À exécuter une seule fois
import subprocess, sys

packages = ["transformers", "sentence-transformers", "scikit-learn", "torch", "accelerate"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("Installation terminée.")

Installation terminée.


---
## Partie 1 - Prompting comparatif

Rappel (séance 4) :
- **Zero-shot** : une instruction seule, sans exemple
- **Few-shot** : quelques exemples résolus avant la vraie question
- **Chain-of-Thought (CoT)** : on demande un raisonnement explicite avant la réponse finale

On utilise ici des questions de **raisonnement par comparaison** (transitivité) : pas de calcul arithmétique, juste suivre une chaîne logique — un bon compromis pour un tout petit modèle.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def generate(prompt, max_new_tokens=60):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Modèle chargé.")

/opt/anaconda3/envs/reddit/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1467.14it/s, Materializing param=model.norm.weight]                             


Modèle chargé.


In [4]:
questions = [
    ("Alice court plus vite que Bob. Bob court plus vite que Claire. Qui court le moins vite ?", "Claire"),
    ("Le vélo coûte plus cher que la trottinette. La trottinette coûte plus cher que le skateboard. Quel est l'objet le moins cher ?", "skateboard"),
    ("Marc est plus grand que Julie. Julie est plus grande que Tom. Qui est le plus grand ?", "Marc"),
    ("Le serveur Atlas répond plus vite que le serveur Boréal. Le serveur Boréal répond plus vite que le serveur Cyclone. Quel serveur est le plus lent ?", "Cyclone"),
    ("Léa a plus de followers que Sam. Sam a plus de followers que Nina. Qui a le moins de followers ?", "Nina"),
    ("Le laptop pèse plus lourd que la tablette. La tablette pèse plus lourd que le smartphone. Quel appareil est le plus léger ?", "smartphone"),
]

print(f"{len(questions)} questions chargées.")

6 questions chargées.


### 1.1 Zero-shot

In [5]:
def build_zero_shot_prompt(question):
    return question + " Réponds uniquement par un nom, sans explication."

resultats_zero_shot = []
for question, attendu in questions:
    reponse = generate(build_zero_shot_prompt(question), max_new_tokens=20)
    resultats_zero_shot.append(reponse)
    print(f"Attendu : {attendu:12s} | Réponse : {reponse[:70]}")

Attendu : Claire       | Réponse : Alice court plus vite que Bob. Bob court plus vite que Claire. Qui cou
Attendu : skateboard   | Réponse : Le vélo coûte plus cher que la trottinette. La trottinet
Attendu : Marc         | Réponse : Marc est le plus grand.
Attendu : Cyclone      | Réponse : Le serveur Atlas est le plus lent que le serveur Boréal.
Attendu : Nina         | Réponse : Sam a plus de followers que Nina.
Attendu : smartphone   | Réponse : Le smartphone est le plus léger.


### 1.2 Few-shot

**Exercice :** Complétez `build_few_shot_prompt(question)` pour qu'elle ajoute les deux exemples résolus (`FEW_SHOT_PREFIX`) **avant** la question, suivie de `"Réponse :"` (sans y répondre — c'est au modèle de continuer).

In [ ]:
FEW_SHOT_PREFIX = '''Voici des exemples de raisonnement par comparaison :

Question : Le chat est plus rapide que la tortue. La tortue est plus rapide que l'escargot. Qui est le plus lent ?
Réponse : escargot

Question : Le livre coûte plus cher que le stylo. Le stylo coûte plus cher que la gomme. Quel objet est le moins cher ?
Réponse : gomme

'''

def build_few_shot_prompt(question):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
resultats_few_shot = []
for question, attendu in questions:
    reponse = generate(build_few_shot_prompt(question), max_new_tokens=20)
    resultats_few_shot.append(reponse)
    print(f"Attendu : {attendu:12s} | Réponse : {reponse[:70]}")

### 1.3 Chain-of-Thought

In [ ]:
def build_cot_prompt(question):
    return question + " Réfléchis étape par étape avant de répondre, puis conclus par 'Réponse : <nom>'."

resultats_cot = []
for question, attendu in questions:
    reponse = generate(build_cot_prompt(question), max_new_tokens=100)
    resultats_cot.append(reponse)
    print(f"Attendu : {attendu:12s} | Réponse : {reponse[:100]}")

### 1.4 Comparaison des trois stratégies

**Exercice :** Complétez `correct(reponse, attendu)`, qui renvoie `True` si le mot attendu apparaît dans la réponse générée (**insensible à la casse**).

In [ ]:
def correct(reponse, attendu):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
strategies = {
    "Zero-shot": resultats_zero_shot,
    "Few-shot": resultats_few_shot,
    "Chain-of-Thought": resultats_cot,
}

print(f"{'Stratégie':<18} | Exactitude")
print("-" * 32)
for nom, resultats in strategies.items():
    n_correct = sum(correct(r, attendu) for r, (_, attendu) in zip(resultats, questions))
    print(f"{nom:<18} | {n_correct}/{len(questions)}")

**Question de réflexion :** Le Chain-of-Thought améliore-t-il nettement les résultats ici ? Le papier de Wei et al. (2022, séance 4) note que les gains du CoT apparaissent surtout **au-delà d'une certaine échelle de modèle** (dizaines de milliards de paramètres). Avec un modèle de 360M, est-ce cohérent avec ce que vous observez ?

**Votre réponse :**

---
## Partie 2 - Mini-RAG

On utilise une base de connaissances **fictive** (une entreprise imaginaire, "NeoTech") : le modèle n'a **certainement jamais vu** ces informations pendant son pré-entraînement, ce qui permet d'isoler proprement l'effet du RAG.

In [ ]:
documents = [
    "Le service AuthGateway de NeoTech utilise le protocole interne NT-SSO v3, déployé en mars 2024, qui remplace l'ancien système LegacyAuth.",
    "La fonction sanitize_user_payload() du backend NeoTech retire les balises HTML et échappe les caractères SQL avant d'appeler insert_user().",
    "Le budget cloud mensuel de l'équipe Data chez NeoTech est plafonné à 12 400 euros depuis janvier 2025.",
    "Le modèle interne NeoLLM-mini de NeoTech est un modèle de 2,3 milliards de paramètres entraîné en interne pour la classification de tickets support.",
    "La politique de télétravail chez NeoTech autorise 3 jours de télétravail par semaine, sauf pour l'équipe Support qui est en 100% présentiel le mardi.",
    "Le pipeline CI/CD de NeoTech s'appelle ForgeFlow et déploie automatiquement en production toutes les nuits à 2h00.",
]

print(f"{len(documents)} documents chargés.")

In [ ]:
from sentence_transformers import SentenceTransformer

# Même modèle d'embeddings que le lab 2
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
doc_embeddings = embedder.encode(documents)

print("Documents vectorisés. Forme :", doc_embeddings.shape)

### 2.1 Retrieval

**Exercice :** Complétez `retrieve(question, k=1)`, qui renvoie les `k` documents les plus proches de la question (similarité cosinus, comme en lab 2 avec `sklearn.metrics.pairwise.cosine_similarity`).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve(question, k=1):
    # YOUR CODE HERE
    # Indice : encoder la question avec `embedder`, calculer la similarité
    # avec `doc_embeddings`, puis renvoyer les k documents les plus proches.
    raise NotImplementedError

# Test rapide
retrieve("Que fait la fonction sanitize_user_payload chez NeoTech ?")

### 2.2 Générer avec et sans RAG

In [ ]:
rag_questions = [
    "Que fait la fonction sanitize_user_payload chez NeoTech ?",
    "Combien de jours de télétravail sont autorisés chez NeoTech ?",
    "Comment s'appelle le pipeline CI/CD de NeoTech et quand déploie-t-il ?",
]

for question in rag_questions:
    contexte = retrieve(question, k=1)[0]
    prompt_rag = f"Contexte : {contexte}\n\nQuestion : {question}\nRéponds en une phrase, en te basant uniquement sur le contexte."

    print("Q :", question)
    print("SANS RAG :", generate(question, max_new_tokens=60))
    print("AVEC RAG :", generate(prompt_rag, max_new_tokens=60))
    print()

**Observation attendue :** sans RAG, le modèle **invente** une explication plausible mais fausse (il n'a jamais vu ces informations). Avec RAG, la réponse est ancrée dans le document récupéré.

### 2.3 Pour aller plus loin (bonus, optionnel)

- Relancez avec `k=2` sur la question sur `sanitize_user_payload` : le modèle mélange-t-il les deux documents récupérés dans sa réponse ?
- Posez une question **hors du corpus** (ex. *"Quelle est la couleur du logo de NeoTech ?"*) avec RAG activé : que se passe-t-il quand aucun document pertinent n'existe ?

---
## Récapitulatif

Remplissez le tableau ci-dessous (oui / non / partiellement) à partir de vos observations :

| | Utilise des exemples | Explicite un raisonnement | Ancre la réponse dans des données externes |
|---|---|---|---|
| **Zero-shot** | ? | ? | ? |
| **Few-shot** | ? | ? | ? |
| **Chain-of-Thought** | ? | ? | ? |
| **RAG** | ? | ? | ? |

**Pour aller plus loin (non couvert dans ce lab, voir séance 4 section 5) :** RAG et prompting n'aident le modèle qu'à mieux utiliser ce qu'il sait déjà ou ce qu'on lui fournit — pour agir sur le monde extérieur (appeler une vraie API, exécuter du code), il faut le **function calling**.